In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from sentence_transformers import CrossEncoder

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

model_name = "cross-encoder/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
model = CrossEncoder(model_name, device=device)
pair_inputs = list(zip(df["sentence1"].tolist(), df["sentence2"].tolist()))

print({
    "model_name": model_name,
    "num_pairs": len(pair_inputs),
    "device": device,
})

In [ ]:
batch_latencies = []
pred_batches = []

for start_idx in range(0, len(pair_inputs), batch_size):
    batch_pairs = pair_inputs[start_idx:start_idx + batch_size]
    t0 = time.time()
    batch_preds = model.predict(batch_pairs, batch_size=len(batch_pairs), show_progress_bar=False)
    batch_latencies.append(time.time() - t0)
    pred_batches.append(np.asarray(batch_preds, dtype=np.float32))

predicted_score_0_5 = np.concatenate(pred_batches, axis=0)
labels = df["label"].to_numpy(dtype=np.float32)
absolute_error = np.abs(predicted_score_0_5 - labels)

results_df = df.copy()
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error

print(results_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

batch_sizes_seen = [min(batch_size, len(pair_inputs) - i) for i in range(0, len(pair_inputs), batch_size)]
latency_df = pd.DataFrame({
    "batch_index": np.arange(len(batch_latencies), dtype=np.int32),
    "batch_size": batch_sizes_seen,
    "latency_seconds": batch_latencies,
})
latency_df["examples_per_second"] = latency_df["batch_size"] / latency_df["latency_seconds"]

error_ranked_df = results_df.sort_values(
    by=["absolute_error", "label", "predicted_score_0_5"],
    ascending=[False, False, False],
).reset_index(drop=True)

print(latency_df.head(10))
print(error_ranked_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"num_batches: {len(batch_latencies)}")
print(f"mean_batch_latency_seconds: {float(np.mean(batch_latencies)):.6f}")
print(f"median_batch_latency_seconds: {float(np.median(batch_latencies)):.6f}")
print(f"min_batch_latency_seconds: {float(np.min(batch_latencies)):.6f}")
print(f"max_batch_latency_seconds: {float(np.max(batch_latencies)):.6f}")
print(f"mean_examples_per_second: {float(latency_df['examples_per_second'].mean()):.6f}")
print(f"median_examples_per_second: {float(latency_df['examples_per_second'].median()):.6f}")
print("highest_error_examples:")
print(error_ranked_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error"]].head(10).to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")